# Lab 5 - Multiagent Pattern for Transmission and Custody Review

This is the main Lab 5 forensic notebook. If you want a gentler introduction first, start with [03a_multiagent_warmup.ipynb](./03a_multiagent_warmup.ipynb), which uses the same Multiagent Pattern in a non-forensics setting.

**Case question:** Was `patients_contacts.png` transmitted from the outreach phone? Use the available evidence to assign a confidence label (`confirmed`, `likely`, or `unconfirmed`) and explain whether the missing transfer in the chain-of-custody log weakens confidence in that conclusion.

In this notebook, the three agent roles match the Lab 5 case more directly:
- `InvestigationAgent` reconstructs the technical timeline.
- `EvidenceVerificationAgent` checks whether the technical claims are actually supported by the artifacts.
- `CustodyAuditAgent` reviews the chain-of-custody record and writes the final evidence-bounded conclusion.


![Figure 1. Multiagent-pattern workflow for Lab 5](./figures/lab5_multiagent_workflow.svg)

This figure shows the Lab 5 workflow: the case package is split across specialized roles, the agent outputs are compared, and the final conclusion is only written after both the technical evidence and the evidence-handling record are reviewed.

## What You Will Do
1. Set up the notebook and point it to the Lab 5 artifact files.
2. Define simple case-specific tools for each artifact type.
3. Inspect the artifact list and a few example tool outputs.
4. Create three specialized agents with clear role boundaries.
5. Run the agents step by step so you can see how context moves across the team.
6. Let `CustodyAuditAgent` write the final confidence-bounded case conclusion.
7. Recreate the same dependency graph with `Crew` and run the full workflow again.


## Role Outputs and Context

Keep these role-specific outputs visible as you work:
- `InvestigationAgent` should return: `Investigation timeline`, `Initial transmission assessment`, `Open technical questions`.
- `EvidenceVerificationAgent` should return: `Claims checked`, `Supported evidence`, `Unsupported or missing evidence`, `Revised transmission rating`.
- `CustodyAuditAgent` should return: `Verified technical finding`, `Custody review`, `Confidence impact`, `Final conclusion`.

Important: downstream agents automatically receive the earlier agents' outputs as context. That context acts like short-term working notes inside the multiagent workflow.


## Part A — Build and Run the Forensic Collaboration Manually

In Part A, you will set up the notebook, inspect the case tools, create the three forensic agents, and run them one at a time. This keeps the collaboration visible so you can see exactly what each role contributes before the packaged `Crew` version appears.


### Step 1: Set Up the Notebook

This step prepares Python to access the Lab 5 tools, agents, and staged case artifacts.

- **Inputs:** the Lab 5 folder, its `.env` file, the repository's `src/` directory, and the `data/` folder.
- **Processing:** load the model settings, import the course classes, and locate the artifact files.
- **Output:** a ready notebook environment for the forensic evidence and agent steps.


In [ ]:
# Purpose: This cell supports Step 1: Set Up the Notebook by loading the libraries, settings, and data needed for this section.
import csv
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

LAB_NAME = 'lab5_multiagent_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(f'Open this notebook from the {LAB_NAME} folder.')

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError('Expected .env in this folder. Copy .env.example to .env first.')

src_dir = repo_root / 'src'
if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

load_dotenv(env_path, override=True)

MODEL = os.getenv('MODEL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f'MODEL or OLLAMA_BASE_URL is missing from {env_path}')

data_dir = lab_dir / 'data'
if not data_dir.exists():
    raise FileNotFoundError('Expected a data/ folder in this lab directory.')

# Import the course-specific classes only after src/ has been added to the Python path.
from agentic_patterns.multiagent_pattern.agent import Agent
from agentic_patterns.multiagent_pattern.crew import Crew
from agentic_patterns.tool_pattern.tool import tool

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)
print('Data folder:', data_dir)
print('Model:', MODEL)


### Step 2: Define Small Evidence Tools

This step creates the small, case-specific tools the agents use to inspect the evidence.

- **Inputs:** the staged CSV files in `data/`.
- **Processing:** wrap each artifact type in a simple tool: technical tools read device, file, messaging, and network events; the custody tool reads the evidence-handling record.
- **Output:** `technical_tools` and `custody_tools`, which keep the technical transmission question separate from the custody question. A missing entry in `chain_of_custody.csv` documents an analyst-to-analyst handoff gap; it does not prove whether the phone completed a transmission.


In [ ]:
# Purpose: This cell supports Step 2: Define Small Evidence Tools by defining reusable helper code that performs the work described here.
def read_csv_rows(filename: str):
    # Read one CSV file into a list of small dictionaries so the notebook can show structured evidence.
    with (data_dir / filename).open(newline='') as handle:
        return list(csv.DictReader(handle))


@tool
def list_artifact_files():
    """Return the available artifact files for Lab 5."""
    return sorted(path.name for path in data_dir.iterdir() if path.is_file())


@tool
def get_device_state_events():
    """Return the lock and unlock records that bound the unattended interval."""
    return read_csv_rows('device_state.csv')


@tool
def get_patients_file_event():
    """Return the file event for patients_contacts.png."""
    return read_csv_rows('file_events.csv')


@tool
def get_securechat_attachment_event():
    """Return the SecureChat attachment-attempt record for patients_contacts.png."""
    return read_csv_rows('messaging_events.csv')


@tool
def get_securechat_network_event():
    """Return the SecureChat network upload record."""
    return read_csv_rows('network_events.csv')


@tool
def get_chain_of_custody_events():
    """Return the chain-of-custody records used to judge evidence-handling completeness."""
    return read_csv_rows('chain_of_custody.csv')


technical_tools = [
    list_artifact_files,
    get_device_state_events,
    get_patients_file_event,
    get_securechat_attachment_event,
    get_securechat_network_event,
]

custody_tools = [
    list_artifact_files,
    get_chain_of_custody_events,
]


### Step 3: Inspect the Artifact List and Example Tool Outputs

This step shows representative evidence before the agents begin their review.

- **Inputs:** the artifact-list tool plus example messaging, network, and custody tool calls.
- **Processing:** run the tools directly and display their results; no agent reasoning occurs yet.
- **Output:** an initial view of the records. Notice that messaging and network records describe device activity during the incident window, while the custody record describes later analyst handling.


In [ ]:
# Purpose: This cell supports Step 3: Inspect the Artifact List and Example Tool Outputs by running the model or agent action and saving its result for review.
artifact_files = list_artifact_files.run()
attachment_example = get_securechat_attachment_event.run()
custody_example = get_chain_of_custody_events.run()

display(Markdown('### Artifact Files\n\n```json\n' + json.dumps(artifact_files, indent=2) + '\n```'))
display(Markdown('### Example Messaging Record\n\n```json\n' + json.dumps(attachment_example, indent=2) + '\n```'))
display(Markdown('### Example Custody Record\n\n```json\n' + json.dumps(custody_example, indent=2) + '\n```'))


### Step 4: Create the Three Forensic Agents

This step creates a specialized team that can build, test, and qualify a forensic conclusion.

- **Inputs:** the shared `LAB_QUESTION`, `CASE_SCOPE`, and the technical and custody tools.
- **Processing:** configure `InvestigationAgent` to build the technical account, `EvidenceVerificationAgent` to test it, and `CustodyAuditAgent` to review the handling record. Connect them so later roles receive the earlier findings as context.
- **Output:** three connected agents with distinct responsibilities. Later agents should challenge or narrow earlier claims when the records do not support them.


In [ ]:
# Purpose: This cell supports Step 4: Create the Three Forensic Agents by running the model or agent action and saving its result for review.
# Shared case context: every agent receives the same question and case facts before adding role-specific instructions.
LAB_QUESTION = (
    'Was patients_contacts.png transmitted from the outreach phone, should the conclusion be labeled '
    'confirmed, likely, or unconfirmed, and does the missing transfer in the chain-of-custody log weaken '
    'confidence in that conclusion?'
)

CASE_SCOPE = """
Case facts:
- Incident window: 2026-03-01T18:35:00Z to 2026-03-01T19:10:00Z.
- File of interest: patients_contacts.png.
- Technical artifacts include device-state, file, messaging, and network records.
- Evidence-handling review uses chain_of_custody.csv.
- In this lab, the missing transfer refers to an undocumented custody handoff of the working copy or exported artifacts from one analyst to another analyst, not proof about whether the phone completed transmission.
- Final conclusion must use one label: confirmed, likely, or unconfirmed.
""".strip()


def build_forensic_agents():
    investigation_agent = Agent(
        name='InvestigationAgent',
        backstory=(
            'You reconstruct mobile-forensics timelines. '
            'Focus on ordering technical events carefully and avoid overclaiming upload completion.'
        ),
        task_description=(
            f'{LAB_QUESTION}\n\n'
            f'{CASE_SCOPE}\n\n'
            'Use the technical evidence tools to reconstruct the timeline and produce an initial transmission assessment.'
        ),
        task_expected_output=(
            'Use these labels:\n'
            'Investigation timeline:\n'
            'Initial transmission assessment:\n'
            'Open technical questions:'
        ),
        tools=technical_tools,
        llm=MODEL,
    )

    evidence_verification_agent = Agent(
        name='EvidenceVerificationAgent',
        backstory=(
            'You verify whether technical claims are supported by specific artifact records. '
            'You correct unsupported claims and keep the rating evidence-bounded.'
        ),
        task_description=(
            f'{LAB_QUESTION}\n\n'
            f'{CASE_SCOPE}\n\n'
            'Use the technical evidence tools to test the earlier technical claims. '
            'If the artifacts show only file creation, an attach attempt, and upload_started, say so clearly. '
            'Do not treat upload_started as confirmed completion.'
        ),
        task_expected_output=(
            'Use these labels:\n'
            'Claims checked:\n'
            'Supported evidence:\n'
            'Unsupported or missing evidence:\n'
            'Revised transmission rating:'
        ),
        tools=technical_tools,
        llm=MODEL,
    )

    custody_audit_agent = Agent(
        name='CustodyAuditAgent',
        backstory=(
            'You review chain-of-custody records and explain how evidence-handling gaps affect final confidence. '
            'You use the earlier agents\' technical findings but focus on whether the handling record supports trust.'
        ),
        task_description=(
            f'{LAB_QUESTION}\n\n'
            f'{CASE_SCOPE}\n\n'
            'Use the chain-of-custody tool and the earlier agent context to write the final case conclusion. '
            'Check whether the chain-of-custody record is missing a handoff entry between evidence handlers. '
            'Treat that custody gap as an evidence-handling issue, not as proof about whether the phone completed transmission. '
            'Use the earlier agents\' technical findings as context rather than reconstructing the technical timeline from scratch. '
            'Explain how the custody gap affects confidence in the final label (confirmed, likely, or unconfirmed) while keeping the conclusion evidence-bounded.'
        ),
        task_expected_output=(
            'Use these labels:\n'
            'Verified technical finding:\n'
            'Custody review:\n'
            'Confidence impact:\n'
            'Final conclusion:'
        ),
        tools=custody_tools,
        llm=MODEL,
    )

    investigation_agent >> evidence_verification_agent
    investigation_agent >> custody_audit_agent
    evidence_verification_agent >> custody_audit_agent

    return investigation_agent, evidence_verification_agent, custody_audit_agent


def run_crew_with_markdown(crew: Crew):
    outputs = {}

    # Crew chooses a safe dependency order; we only improve the notebook display format.
    for agent in crew.topological_sort():
        output = agent.run()
        outputs[agent.name] = output
        display(Markdown(f'### {agent.name} Output\n\n{output}'))

    return outputs


investigation_agent, evidence_verification_agent, custody_audit_agent = build_forensic_agents()

print('InvestigationAgent dependents:', investigation_agent.dependents)
print('EvidenceVerificationAgent dependencies:', evidence_verification_agent.dependencies)
print('CustodyAuditAgent dependencies:', custody_audit_agent.dependencies)


### Step 5: Run `InvestigationAgent`

This step produces the initial technical account of the possible transmission.

- **Inputs:** the shared case question, `InvestigationAgent`'s role instructions, and the technical evidence tools.
- **Processing:** inspect the device, file, messaging, and network records to organize the timeline.
- **Output:** a preliminary transmission assessment and open technical questions. Treat file creation, attachment, and `upload_started` as evidence of an attempt, not confirmed completion.


In [ ]:
# Purpose: This cell supports Step 5: Run `InvestigationAgent` by running the model or agent action and saving its result for review.
investigation_output = investigation_agent.run()
display(Markdown('### InvestigationAgent Output\n\n' + investigation_output))


### Step 6: Inspect and Run `EvidenceVerificationAgent`

This step tests the initial technical account against the underlying records.

- **Inputs:** the investigation output passed as context, the shared case question, and the technical evidence tools.
- **Processing:** display the handoff, then run `EvidenceVerificationAgent` to compare the earlier claims with the files and logs.
- **Output:** a revised transmission rating that identifies supported evidence and missing evidence. Lower the conclusion when the artifacts show an upload attempt but not completion.


In [ ]:
# Purpose: This cell supports Step 6: Inspect and Run `EvidenceVerificationAgent` by showing the result in a form that is easier to review.
display(Markdown('### EvidenceVerificationAgent Received Context\n\n```text\n' + evidence_verification_agent.context + '\n```'))

evidence_verification_output = evidence_verification_agent.run()
display(Markdown('### EvidenceVerificationAgent Output\n\n' + evidence_verification_output))


### Step 7: Inspect and Run `CustodyAuditAgent`

This step adds an evidence-handling review to the verified technical finding.

- **Inputs:** the earlier technical outputs passed as context, the shared case question, and the chain-of-custody tool.
- **Processing:** display the combined handoff, then run `CustodyAuditAgent` to check the documented evidence transfers.
- **Output:** a confidence-bounded final conclusion that explains the custody gap. The missing transfer is a handoff between evidence handlers, not a phone-to-contact transmission.


In [ ]:
# Purpose: This cell supports Step 7: Inspect and Run `CustodyAuditAgent` by showing the result in a form that is easier to review.
display(Markdown('### CustodyAuditAgent Received Context\n\n```text\n' + custody_audit_agent.context + '\n```'))

custody_audit_output = custody_audit_agent.run()
display(Markdown('### CustodyAuditAgent Final Conclusion\n\n' + custody_audit_output))


## Part B — Recreate the Same Workflow with `Crew`

The manual run above makes each role easier to inspect. `Crew` packages the same idea more neatly by holding the dependency graph in one place.

The next cells create a fresh forensic team inside a `Crew` context, show the dependency graph, and run the same workflow again using `Crew`'s topological order.

### Reminder

Part B changes the orchestration, not the case logic. Students should expect the same cautious reasoning even if the wording changes.


### Step 1: Build the Same Team Inside `Crew`

This step creates a fresh forensic team for the packaged workflow.

- **Inputs:** the `build_forensic_agents` helper and a new `Crew` context.
- **Processing:** rebuild the three agents inside `Crew`, which stores their dependency graph.
- **Output:** a fresh team with the same roles and handoffs used in Part A.


In [ ]:
# Purpose: This cell supports Step 1: Build the Same Team Inside `Crew` by creating and connecting the agents that will complete this workflow.
with Crew() as crew:
    crew_investigation_agent, crew_evidence_verification_agent, crew_custody_audit_agent = build_forensic_agents()


### Step 2: Visualize the Forensic Crew Graph

This step turns the forensic team's dependency rules into a visual map.

- **Inputs:** the `Crew` dependency graph created in Step 1.
- **Processing:** render the graph; no agent evaluates case evidence in this step.
- **Output:** a visual check that investigation comes first, verification tests the technical account, and custody review receives both earlier outputs before the final conclusion.


In [ ]:
# Purpose: This cell supports Step 2: Visualize the Forensic Crew Graph by preparing or examining the evidence used in this section.
crew.plot()


### Step 3: Run the Full Forensic Crew

This step runs the complete forensic team through one coordinated `Crew` call.

- **Inputs:** the fresh `Crew`, its dependency graph, the shared case question, and the staged evidence tools.
- **Processing:** `Crew` runs each agent in dependency order and passes earlier outputs as context; the helper displays each result as Markdown.
- **Output:** the same sequence of role-specific findings as Part A. Wording may vary by run, so judge whether the final conclusion remains evidence-bounded.


In [ ]:
# Purpose: This cell supports Step 3: Run the Full Forensic Crew by preparing or examining the evidence used in this section.
crew_outputs = run_crew_with_markdown(crew)


## What To Notice

As you review the outputs, look for these points:
- Did `InvestigationAgent` keep the technical timeline clear without overstating completion?
- Did `EvidenceVerificationAgent` lower confidence when it saw that the artifacts only show `upload_started`, not `upload_completed`?
- Did `CustodyAuditAgent` explain why a missing transfer entry weakens confidence in the final conclusion?
- Did you keep the technical transmission question separate from the evidence-handling transfer question?

Those three questions capture the main Lab 5 idea: multiagent collaboration is useful when different roles contribute different kinds of evidence checking before a final answer is written.
